# Goal 2：参数实验与顺序比较

本工作 Notebook 对应基础作业的分段、方向去噪、DP 实现空缺及三个选做参数实验，并回答处理顺序问题。结果均是 `source_crs=UNVERIFIED` 条件下的工作平面几何分析；原始记录不自动代表独立用户或完整骑行。

In [ ]:
from pathlib import Path
import sys, json, tempfile, shutil
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'AGENTS.md').is_file() and (p/'task1').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from task1.workflow.io import read_json, digest, object_hash
EV = ROOT/'task1/evidence/goal2'
current = read_json(EV/'current_runs.json')
contract = read_json(ROOT/'task1/config/goal2/contract.json')
split = read_json(EV/'data/split_manifest.json')
MODE = 'RECOMPUTE'
ENABLE_LIVE = False
assert MODE == 'RECOMPUTE' and ENABLE_LIVE is False, '新的 LIVE 调用须使用独立命令并显式启用，默认 Notebook 不发起模型请求。'
WORK = Path(tempfile.mkdtemp(prefix='sc-g2-notebook-'))
def show(rows):
    display(pd.DataFrame(rows))
def figure(name):
    directory = Path(recomputed.get('figure_directory', WORK/'figures'))
    target = directory/(name+'.png')
    assert target.is_file(), f'复算图缺失: {target}'
    display(Image(filename=str(target), width=1050))
print('运行方式:', MODE, '| 新模型调用:', 0)
print('原始来源 datum:', contract['source_crs'], '| 单位: 工作米 / 原始相对秒')
print('当前固定分区:', {name:len(ids) for name,ids in split['splits'].items()})


## 教师入口与实现

| 教师材料 | 本轮入口 |
|---|---|
| 基础 Notebook cell 4：分段 | `workflow/geometry.py:split_trajectory` 与 `filter_segments` |
| cell 6：去噪 | `denoise_trajectory`，一次标记后同时删除 |
| cell 8：简化 | `douglas_peucker_indices`，有限线段与原始索引 |
| cells 12–17：三组参数实验 | `g2_experiments.parameter_development` |
| PPT 42：能否交换顺序 | 六种 S/D/P 的真实受控对照 |

G2 使用单独合同与入口，G1 的七条回归合同保留。S 前不隐藏分段；每阶段重新计算邻接特征。

In [ ]:
from task1.workflow.g2_pipeline import REFERENCE_PARAMETERS, ORDERS
print('参考配置:', REFERENCE_PARAMETERS)
print('已登记顺序:', ORDERS)
print('规定参数网格:')
show([{'parameter':key,'values':value} for key,value in contract['parameter_grids'].items()])

## Restart Kernel → Run All 的实际重算

下面从不可变 raw 重算完整 DEVELOPMENT 参数与顺序产物，以及当前 G2_EVAL 代表配置、冻结单项和可行顺序，重新检查参数、原始身份、阶段操作、完整 P 参考和账本。保存文件只作为版本比较目标。此路径不以加载 CSV 代替实验，也不发起新的 LLM 调用。

In [ ]:
from task1.scripts.goal2 import recompute
recomputed = recompute(topic='parameters', output_directory=WORK, rebuild_figures=True)
assert recomputed['status'] == 'VERIFIED', recomputed
assert recomputed['new_model_calls'] == 0, recomputed
print('从 raw 和冻结提议复算:', recomputed['status'])
print('新增模型调用:', recomputed['new_model_calls'])
print('检查项数量:', len(recomputed['checks']))
show(recomputed['checks'][:20])
print('全部检查结果保存在独立复算输出，不把重复计算计为新独立实验。')


## 分段与短片段过滤

分段在严格大于阈值的边右点前切开；短段过滤单独判点数和段内累计长度。`dt=15` 会切开 20 秒间隔。95 工作米是教学例值，不能解释为本数据已确认限速。图同时给出 raw 点保留与整条无输出，两者分母不同；全量逐记录结果仍在当前 run。

In [ ]:
parameter_run = EV/'runs'/current['parameter_development']
parameter_rows = read_json(parameter_run/'parameter_table.json')
show([{'config_id':row['config_id'],**row['parameters'],**{key:row['summary'][key] for key in ('n_records','n_input','n_final','n_no_output_records','point_retention','common_coverage')}} for row in parameter_rows])
figure('segmentation_filter_response')
figure('segmentation_filter_grids')

## 一次方向删除与 DP 的独立分母

方向不可计算窗口和首尾点保持；删点后的新邻接不触发第二轮迭代。DP 省点率是 `1 − N_P_immediate_output / N_P_input`，不把上游或后续阶段损失算入。共同 raw 几何误差另外评价；各自 clean 的小误差不能证明整体更好。

In [ ]:
figure('direction_threshold_and_neighborhood')
figure('dp_error_compression')

## 顺序是否可以交换

六顺序都按实际当前索引运行。下表将工程正确与共同原始断点、覆盖保护分开；违反保护是有效负结果。P 先执行可能删去断点触发信息，D 先执行可能跨间断取方向窗口，P 先于 D 会改变 D 的邻域。是否进入阶段留出评估由冻结的保护规则和实际失败证据决定。

In [ ]:
order_rows = read_json(EV/'runs'/current['order_development']/'order_table.json')
show([{'order':row['order'],'engineering':row['engineering_status'],'research':row['research_status'],'failed_records':len(row['safety_failures']),**{key:row['summary'][key] for key in ('common_coverage','raw_break_crossings','direction_windows_across_raw_breaks','n_no_output_records')}} for row in order_rows])
figure('order_protection_and_neighborhoods')

## 解释边界

统计单位首先是原始记录。分层样本不代表总体均值；删点数、长度变化和计算可行性不是噪声检测准确率。三单项只形成 Goal 3 备选，本轮不组合或冻结最终方法。

In [ ]:
print('raw hash unchanged:', digest(ROOT/contract['raw_path']) == contract['raw_sha256'])
print('GPT_SECOND_REVIEW=PENDING; Submission=NOT_READY')
shutil.rmtree(WORK)